In [2]:
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/cookie_cats.db")

In [4]:
# -----------------------------
# 1. GROUP SIZES
# -----------------------------
query = """
SELECT
    version,
    COUNT(*) AS players,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM players), 2) AS percentage
FROM players
GROUP BY version;
"""

group_sizes = pd.read_sql(query, con)
print("\n=== GROUP SIZES ===")
print(group_sizes)


# -----------------------------
# 2. DUPLICATE USER IDs
# -----------------------------
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT userid) AS unique_users,
    COUNT(*) - COUNT(DISTINCT userid) AS duplicate_rows
FROM players;
"""

duplicates = pd.read_sql(query, con)
print("\n=== DUPLICATE USER IDS ===")
print(duplicates)


# -----------------------------
# 3. GAME ROUND STATISTICS
# -----------------------------
query = """
SELECT
    COUNT(*) AS players,
    MIN(sum_gamerounds) AS min_rounds,
    MAX(sum_gamerounds) AS max_rounds,
    ROUND(AVG(sum_gamerounds), 2) AS avg_rounds
FROM players;
"""

round_stats = pd.read_sql(query, con)
print("\n=== GAME ROUND STATISTICS ===")
print(round_stats)


# -----------------------------
# 4. EXTREME OUTLIERS
# -----------------------------
query = """
SELECT *
FROM players
ORDER BY sum_gamerounds DESC
LIMIT 20;
"""

outliers = pd.read_sql(query, con)
print("\n=== TOP 20 GAME-ROUND OUTLIERS ===")
print(outliers)


# -----------------------------
# 5. MISSING VALUES
# -----------------------------
query = """
SELECT
    SUM(CASE WHEN userid IS NULL THEN 1 ELSE 0 END) AS missing_userid,
    SUM(CASE WHEN version IS NULL THEN 1 ELSE 0 END) AS missing_version,
    SUM(CASE WHEN sum_gamerounds IS NULL THEN 1 ELSE 0 END) AS missing_gamerounds,
    SUM(CASE WHEN retention_1 IS NULL THEN 1 ELSE 0 END) AS missing_retention_1,
    SUM(CASE WHEN retention_7 IS NULL THEN 1 ELSE 0 END) AS missing_retention_7
FROM players;
"""

missing = pd.read_sql(query, con)
print("\n=== MISSING VALUES ===")
print(missing)


# -----------------------------
# 6. RETENTION BY GROUP
# -----------------------------
query = """
SELECT
    version,
    COUNT(*) AS players,

    SUM(retention_1) AS retained_day1,
    ROUND(100.0 * AVG(retention_1), 2) AS retention_1_pct,

    SUM(retention_7) AS retained_day7,
    ROUND(100.0 * AVG(retention_7), 2) AS retention_7_pct

FROM players
GROUP BY version;
"""

retention = pd.read_sql(query, con)
print("\n=== RETENTION BY GROUP ===")
print(retention)


# -----------------------------
# 7. GROUP-WISE GAME ROUNDS
# -----------------------------
query = """
SELECT
    version,
    COUNT(*) AS players,
    ROUND(AVG(sum_gamerounds), 2) AS avg_rounds,
    MIN(sum_gamerounds) AS min_rounds,
    MAX(sum_gamerounds) AS max_rounds
FROM players
GROUP BY version;
"""

group_rounds = pd.read_sql(query, con)
print("\n=== GAME ROUNDS BY GROUP ===")
print(group_rounds)



=== GROUP SIZES ===
   version  players  percentage
0  gate_30    44700       49.56
1  gate_40    45489       50.44

=== DUPLICATE USER IDS ===
   total_rows  unique_users  duplicate_rows
0       90189         90189               0

=== GAME ROUND STATISTICS ===
   players  min_rounds  max_rounds  avg_rounds
0    90189           0       49854       51.87

=== TOP 20 GAME-ROUND OUTLIERS ===
     userid  version  sum_gamerounds  retention_1  retention_7
0   6390605  gate_30           49854            0            1
1    871500  gate_30            2961            1            1
2   3271615  gate_40            2640            1            0
3   4832608  gate_30            2438            1            1
4   5346171  gate_40            2294            1            1
5   5133952  gate_30            2251            1            1
6   9640085  gate_30            2156            1            1
7   4090246  gate_40            2124            1            1
8   9791599  gate_40            2063   

In [5]:
retention = pd.read_sql("""
SELECT
    version,
    COUNT(*) AS players,
    SUM(retention_1) AS retained_day1,
    ROUND(100.0 * AVG(retention_1), 2) AS day1_retention_pct,
    SUM(retention_7) AS retained_day7,
    ROUND(100.0 * AVG(retention_7), 2) AS day7_retention_pct
FROM players
GROUP BY version
ORDER BY version;
""", con)

print(retention.to_string(index=False))

version  players  retained_day1  day1_retention_pct  retained_day7  day7_retention_pct
gate_30    44700          20034               44.82           8502               19.02
gate_40    45489          20119               44.23           8279               18.20


In [6]:
gate30 = retention[retention["version"] == "gate_30"].iloc[0]
gate40 = retention[retention["version"] == "gate_40"].iloc[0]

day1_gap = gate40["day1_retention_pct"] - gate30["day1_retention_pct"]
day7_gap = gate40["day7_retention_pct"] - gate30["day7_retention_pct"]

print(f"Day-1 retention gap: {day1_gap:.2f} percentage points")
print(f"Day-7 retention gap: {day7_gap:.2f} percentage points")

Day-1 retention gap: -0.59 percentage points
Day-7 retention gap: -0.82 percentage points
